In [1]:
c1 = 100*0.5 #100
c2 = 20
c3 = 50
c4 = 200
c5 = 1000
c6 = 300*1/2 

omega = 1

#LAMBDA = 0.001
NUM_OF_INTERNAL = 60
TOTAL_T = 60
NUM_HIDDEN = 1
MOMENTUM = 0.8
BATCH = 16
K = 5
ITERS = 10000 #10000
sigma0 = "vacc_c1_x0.5" #change this one

In [2]:
birth =  0.000053
death =  0.000033 

alpha =0.00135
sigma= 0.06625
beta1 =0.28702
beta2 =0.15829
beta3 =0.03857
delta1 =0.28844
delta2 =0.28266
delta3 =0.14198
gamma =0.28267
p1 =0.01845
p2 =0.14326
mu =0.0014
sigma1 =0.10177
sigma2 =0.03500
sigma3 =0.09363
sigma4 =0.06189
sigma5 =0.07869
sigma6 =0.06298
sigma7 =0.05991
sigma8 =0.06234

N = 6629870
S0 = 3674149/N
V0 = 2845438/N
E0 = 67789/N
I10 = 12113/N
I20 = 498/N
I30 = 96/N
R0 = 28911/N
D0 = 876/N

init = [S0, V0, E0, I10, I20, I30, R0, D0]

In [3]:
#intercept1 = 0.725
#intercept2 = 0.00155
#intercept3 = -0.01

#a1 = 0
#a2 = -0.0086
#a3 = 0

#b1 = 0
#b2 = 10.815
#b3 = 0

k0 = 0.006
k1 = -0.1341

def p1fn(alpha):
    #return intercept2 + a2 * alpha + b2 * alpha ** 2
    return k0 + k1 * alpha

In [4]:
import time
import numpy as np
import tensorflow as tf
from scipy.stats import multivariate_normal as normal
#import matplotlib.pyplot as plt

tf.keras.backend.set_floatx('float64')
tf.compat.v1.enable_eager_execution()


In [5]:
class Solver(object):

    def __init__(self, seed_var):
        self.valid_size = BATCH
        self.batch_size = BATCH
        
        self.num_iterations = ITERS
        self.print_frequency = 200
        self.lr_values = [1e-3, 1e-4, 1e-6]
        self.lr_boundaries = [5000, 8000]
        self.config = Config()

        self.model = WholeNet(seed_var)
        lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(self.lr_boundaries, self.lr_values)
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, epsilon=1e-7)

    def train(self):
        """Training the model"""
        start_time = time.time()
        training_history = []
        control_output = [] #new
        
        dW = self.config.sample(self.valid_size)
        valid_data = dW
        for step in range(self.num_iterations+1):
            if step % self.print_frequency == 0:
                cost, row_out0 = self.model(valid_data, training=True)
                elapsed_time = time.time() - start_time
                training_history.append([step, cost, elapsed_time])
            if step == self.num_iterations:
                cost1, row_out1 = self.model(valid_data, training=True)
                row_output = row_out1.numpy()
                control_output.append(row_output)
                
                print("step: %5u, Cost: %.4e, elapsed time: %3u" % (step, cost, elapsed_time))
            self.train_step(self.config.sample(self.batch_size))
        self.training_history = training_history
        self.control_output = control_output

    @tf.function
    def train_step(self, train_data):
        """Updating the gradients"""
        with tf.GradientTape(persistent=True) as tape:
            loss = self.model(train_data, training = True)
        grad = tape.gradient(loss, self.model.trainable_variables)
        del tape
        self.optimizer.apply_gradients(zip(grad, self.model.trainable_variables))

In [6]:
class WholeNet(tf.keras.Model):
    """Building the neural network architecture"""
    def __init__(self,seed_var):
        super(WholeNet, self).__init__()
        self.config = Config()
        self.subnet = [FNNet(seed_var) for _ in range(self.config.num_time_interval)]

    def call(self, dw, training):
        x_init = tf.ones([1, self.config.dim_x], dtype=tf.dtypes.float64) * init
        time_stamp = np.arange(0, self.config.num_time_interval) * self.config.delta_t
        all_ones = tf.ones([tf.shape(dw)[0], 1], dtype=tf.dtypes.float64)

        # initial state
        x = tf.matmul(all_ones, x_init)

        # initial cost functional
        l = 0.0 
        u_set = []
        for t in range(0, self.config.num_time_interval):
            u = self.subnet[t](x, training)
            u_set.append(tf.stack([u], axis=0))
            
            l = l + self.config.f_fn(time_stamp[t], x, u) * self.config.delta_t
            x = x + self.config.b_fn(time_stamp[t], x, u) * self.config.delta_t + self.config.sigma_fn(time_stamp[t], x, u) * dw[:,:,  t]

        # view the control variable
        control_u = tf.stack(u_set, axis=0)
        # Step 1: Squeeze the tensor to remove dimensions with size 1
        #tensor_2d = tf.squeeze(control_u, axis=[1, 3])
        # Step 2: Calculate the average of each column
        #row_ave1 = tf.reduce_mean(control_u, axis=1)
        
        row_ave2 = tf.reduce_mean(control_u, axis=2)
        # Step 3: Print the column averages
        #print("u timepath1", row_ave1)
        
        print("u timepath2", row_ave2)
        
        row_out1 = tf.squeeze(row_ave2, axis=[1,2])
        
        

        #testformat= ['%.5e']
        #np.savetxt('./Han&E_control output.csv', test1.numpy(), fmt=testformat, delimiter=',')
        
        # terminal condition
        l = l + self.config.h_fn(self.config.total_time, x)
        cost = tf.reduce_mean(l) 

        return cost, row_out1


In [7]:
class FNNet(tf.keras.Model):
    def __init__(self, seed_var):
        super(FNNet, self).__init__()
        self.config = Config()

        tf.random.set_seed(seed_var)
        np.random.seed(seed_var)
        
        num_hiddens = [self.config.dim_x+NUM_HIDDEN, self.config.dim_x+NUM_HIDDEN, self.config.dim_x+NUM_HIDDEN]
        self.bn_layers = [
            tf.keras.layers.BatchNormalization(
                momentum=MOMENTUM,
                epsilon=1e-6,
                beta_initializer=tf.random_normal_initializer(0.0, stddev=0.1,seed=seed_var),
                gamma_initializer=tf.random_uniform_initializer(0.1, 0.5,seed=seed_var)
            )
            for _ in range(len(num_hiddens) + 2)]
        self.dense_layers = [tf.keras.layers.Dense(num_hiddens[i],
                                                   use_bias=True,
                                                   activation='tanh')
                             for i in range(len(num_hiddens))]
        
        # final output should be gradient of size dim_u
        self.dense_layers.append(tf.keras.layers.Dense(self.config.dim_u, activation='sigmoid'))

    def call(self, x, training):
        x = self.bn_layers[0](x, training)
        for i in range(len(self.dense_layers) - 1):
            x = self.dense_layers[i](x)
            x = self.bn_layers[i+1](x, training)
            x = tf.nn.tanh(x)
        x = self.dense_layers[-1](x)
        x = self.bn_layers[-1](x, training)
        return tf.nn.tanh(x) * 0.012 + 0.016

In [8]:
class Config(object):
    """function config"""
    def __init__(self):
        super(Config, self).__init__()
        self.dim_x = 8
        self.dim_u = 1
        self.num_time_interval = NUM_OF_INTERNAL
        self.total_time = TOTAL_T
        self.delta_t = (self.total_time + 0.0) / self.num_time_interval
        self.sqrth = np.sqrt(self.delta_t)
        self.t_stamp = np.arange(0, self.num_time_interval) * self.delta_t

    # sample function
    def sample(self, num_sample):
        dw_sample = normal.rvs(size=[num_sample, self.dim_x, self.num_time_interval]) * self.sqrth
        return dw_sample

    # cost functional integral item
    def f_fn(self, t, x, u):
        tensors = []
        for i in range(BATCH):
            S = x[i][0]
            V = x[i][1]
            E = x[i][2]
            I1 = x[i][3]
            I2 = x[i][4]
            I3 = x[i][5]
            R = x[i][6]
            D = x[i][7]
            alpha = u[i][0]  
            cost = (c1*alpha**2 + c2*E + (c3*I1+c4*I2+c5*I3) + c6*(tf.maximum(tf.cast(0, tf.float64), 1-(S+V+R)))**omega) #*N
            combined_cost = tf.stack([cost], axis=0)
            tensors.append(combined_cost)
            combined_tensor = tf.stack(tensors, axis=0)
        return tf.reduce_sum(combined_tensor, 1, keepdims=True)
        #return tf.reduce_sum(u**2, 1, keepdims=True)

    # cost fuctional terminal function
    def h_fn(self, t, x):
        return 0

    def b_fn(self, t, x, u):
        #return tf.sin(u)
        tensors = []
        for i in range(BATCH):
            S = x[i][0]
            V = x[i][1]
            E = x[i][2]
            I1 = x[i][3]
            I2 = x[i][4]
            I3 = x[i][5]
            R = x[i][6]
            D = x[i][7]
            alpha = u[i][0]
            dS = birth + (- ( beta1 * I1 + beta2 * I2 + beta3 * I3) * S  - alpha * S) - death*S
            dV = (alpha * S - sigma * (beta1 * I1 + beta2 * I2 + beta3 * I3) * V) - death*V
            dE = ((beta1 * I1 + beta2 * I2 + beta3 * I3) * S  + sigma * (beta1 * I1 + beta2 * I2 + beta3 * I3) * V - gamma * E) - death*E
            dI1 = (gamma * E - (delta1 + p1fn(alpha)) * I1) - death*I1
            dI2 = (p1fn(alpha) * I1 - (delta2 + p2) * I2) - death*I2
            dI3 = (p2 * I2 - (delta3 + mu) * I3) - death*I3
            dR = (delta1 * I1+ delta2 * I2 + delta3 * I3) - death*R
            dD = (mu * I3) 
            combined_ten = tf.stack([dS, dV,dE,dI1,dI2,dI3,dR,dD], axis=0)
            tensors.append(combined_ten)
            combined_tensor = tf.stack(tensors, axis=0)
        return combined_tensor

    def sigma_fn(self, t, x, u):
        #return x
        tensors = []
        for i in range(BATCH):
            S = x[i][0]
            V = x[i][1]
            E = x[i][2]
            I1 = x[i][3]
            I2 = x[i][4]
            I3 = x[i][5]
            R = x[i][6]
            D = x[i][7]
            sig_S = sigma1*S
            sig_V = sigma2*V
            sig_E = sigma3*E
            sig_I1 = sigma4*I1
            sig_I2 = sigma5*I2
            sig_I3 = sigma6*I3
            sig_R = sigma7*R
            sig_D = sigma8*D
            combined_sig = tf.stack([sig_S, sig_V,sig_E,sig_I1,sig_I2,sig_I3,sig_R,sig_D], axis=0)
            tensors.append(combined_sig)
            combined_tensor = tf.stack(tensors, axis=0)
        return combined_tensor

In [9]:
def main():
    solver = Solver(seed_var=12345)
    print('Training time %3u:' % (1))
    solver.train()
    data = np.array(solver.training_history)
    control_output_data = np.array(solver.control_output)
    k = K
    
    output = np.zeros((len(data[:, 0]), 2 + k))
    output[:, 0] = data[:, 0]
    output[:, 1] = data[:, 2]
    output[:, 2] = data[:, 1]
    
    control_outs = np.zeros((k, len(control_output_data[0])))
    control_outs[0] = control_output_data[0]
    
    for i in range(k - 1):
        print('Training time %3u:' % (i + 2))
        solver = Solver(seed_var=12345+20+i)
        solver.train()
        data = np.array(solver.training_history)
        control_output_data = np.array(solver.control_output)
        output[:, 3 + i] = data[:, 1]
        control_outs[1+i] = control_output_data[0]

    a = ['%d', '%.5e']
    for _ in range(k):
        a.append('%.5e')
    
    
    #print(control_outs)
    con_format = ['%.5e']
    for _ in range(len(control_output_data[0])-1):
        con_format.append('%.5e')
        
    np.savetxt('./Han&E_Opt_Vacc_'+str(sigma0)+'.csv', control_outs, fmt=con_format, delimiter=',')
    
    np.savetxt('./Han&E_Loss_'+str(sigma0)+'.csv', output, fmt=a, delimiter=',')

    
    print('Solving is done!')

if __name__ == '__main__':
    main()


2024-09-18 06:45:17.126378: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2024-09-18 06:45:17.126618: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Training time   1:
u timepath2 tf.Tensor(
[[[0.01523925]]

 [[0.01528768]]

 [[0.01528301]]

 [[0.01527058]]

 [[0.01530587]]

 [[0.01530268]]

 [[0.01532223]]

 [[0.01529434]]

 [[0.01529826]]

 [[0.01532975]]

 [[0.0153469 ]]

 [[0.01527898]]

 [[0.01532065]]

 [[0.01522362]]

 [[0.01526427]]

 [[0.0153397 ]]

 [[0.01530672]]

 [[0.01531678]]

 [[0.01532883]]

 [[0.0152142 ]]

 [[0.01531378]]

 [[0.01529509]]

 [[0.01524833]]

 [[0.01522942]]

 [[0.01533804]]

 [[0.01537736]]

 [[0.01530415]]

 [[0.01531928]]

 [[0.01531778]]

 [[0.01533714]]

 [[0.01526517]]

 [[0.01527055]]

 [[0.0152873 ]]

 [[0.01528364]]

 [[0.0152013 ]]

 [[0.01527756]]

 [[0.01530469]]

 [[0.01532049]]

 [[0.01537998]]

 [[0.01515832]]

 [[0.01521369]]

 [[0.01527913]]

 [[0.01527517]]

 [[0.01522161]]

 [[0.01536689]]

 [[0.01535955]]

 [[0.01537747]]

 [[0.01531726]]

 [[0.01532235]]

 [[0.01536913]]

 [[0.01535747]]

 [[0.0152192 ]]

 [[0.01544005]]

 [[0.01527224]]

 [[0.01538491]]

 [[0.01525385]]

 [[0.0

2024-09-18 06:49:51.024830: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)


u timepath2 tf.Tensor(
[[[0.0174308 ]]

 [[0.0173252 ]]

 [[0.01738506]]

 [[0.01728953]]

 [[0.01735988]]

 [[0.01725628]]

 [[0.01733726]]

 [[0.01729777]]

 [[0.01731798]]

 [[0.01724298]]

 [[0.01725556]]

 [[0.0172883 ]]

 [[0.01731342]]

 [[0.0171672 ]]

 [[0.01717818]]

 [[0.01715392]]

 [[0.01717955]]

 [[0.01706135]]

 [[0.01700685]]

 [[0.01700534]]

 [[0.01701467]]

 [[0.01704814]]

 [[0.0169829 ]]

 [[0.01697369]]

 [[0.01700561]]

 [[0.01689578]]

 [[0.01676809]]

 [[0.01687685]]

 [[0.01671242]]

 [[0.01667021]]

 [[0.01654079]]

 [[0.01652987]]

 [[0.0165313 ]]

 [[0.01646721]]

 [[0.01642175]]

 [[0.01629383]]

 [[0.01633142]]

 [[0.01611774]]

 [[0.01629631]]

 [[0.0159871 ]]

 [[0.01591273]]

 [[0.01595219]]

 [[0.01573477]]

 [[0.01581187]]

 [[0.01570537]]

 [[0.01558519]]

 [[0.01564943]]

 [[0.01526108]]

 [[0.01529889]]

 [[0.01502727]]

 [[0.01512062]]

 [[0.01480463]]

 [[0.01468533]]

 [[0.01443008]]

 [[0.01389478]]

 [[0.01357329]]

 [[0.01326759]]

 [[0.013

In [10]:
# Create the tensor directly from the data
#data = [0.02075684, 0.01786812]
#tensor = tf.convert_to_tensor(data, dtype=tf.float64)

# Reshape the tensor to have the desired shape (2, 1, 1)
#tensor = tf.reshape(tensor, shape=(2, 1, 1))

#print(tensor)

#tf.compat.v1.enable_eager_execution()
#print(tensor.numpy())



In [11]:
#import tensorflow as tf
#import numpy as np

#test1 = tf.squeeze(tensor, axis=[1,2]).numpy()
#print('test1.numpy',test1)
#
#control_test = []
#for i in range(2):
#    control_test.append(test1[i])
#print('control test',control_test)

    
#testformat= ['%.5e']
#np.savetxt('./Han&E_control output.csv', test1, fmt=testformat, delimiter=',')

In [12]:
#test1 = tf.squeeze(test, axis=[1,2]).numpy()
#outtest = [] 
#outtest.append(test1)
#print(outtest)
#out_array = np.array(outtest)
#print(out_array)


#control_outs = np.zeros((5, len(out_array[0])))
#print(control_outs)
#control_outs[2] = out_array[0]
#print(control_outs)


#np.savetxt('./Han&E_Control_01.csv', control_outs, fmt=con_format, delimiter=',')